# CLOB basics

Gamma tells you what exists. The Central Limit Order Book tells you what it
costs. Everything in this notebook is unauthenticated and read-only, so no
keys are needed.

Prices move, so the numbers below are whatever the market was doing when the
notebook last ran.

In [2]:
import sys
from pathlib import Path

# Run straight from a clone, without installing the package first.
src = Path.cwd().parent / "src"
if src.is_dir():
    sys.path.insert(0, str(src))

import pyolymarket as pyoly

## Picking a live market

The order book only says something interesting about a market that is still
trading, so rather than hard-coding a slug that will eventually close, pull a
few Bitcoin events and take the first market that is open and has token ids.

In [3]:
events = pyoly.polymarket_search_event("bitcoin", results = 5)
market = next(m for e in events for m in e.markets if m.token_ids and not m.resolved)

print(market.data["question"])
print(market.outcomes)

Will Bitcoin reach $100,000 in August?
['Yes', 'No']


## Quotes through the Market object

`Market` forwards these straight to the CLOB, resolving the outcome label to a
token id on the way. The default outcome is index 0.

In [3]:
print("midpoint  ", market.midpoint("Yes"))
print("spread    ", market.spread("Yes"))
print("best bid  ", market.price("BUY", "Yes"))
print("best ask  ", market.price("SELL", "Yes"))

midpoint   0.0125


spread     0.001


best bid   0.012


best ask   0.013


`side` names the side of the book you are reading, not the trade you intend to
make: "BUY" is the best resting buy order, which is the highest bid, and
"SELL" is the best resting sell order, the lowest ask. So the price you would
pay to acquire a token is the "SELL" one. Anything other than those two
strings raises rather than being silently sent to the API.

In [4]:
try:
    market.price("buy sideways")
except ValueError as error:
    print(error)

Unrecognized side: 'buy sideways'. side must be "BUY" or "SELL"


## The order book

`book()` returns the whole book for one outcome. Bids and asks come back
sorted best-first by the wrapper; the API's own ordering is not consistent, so
`book["bids"][0]` is only the best bid because of that sort.

In [5]:
book = market.book("Yes")

print("bids", book["bids"][:3])
print("asks", book["asks"][:3])
print("tick size", book["tick_size"], "| min order", book["min_order_size"])

bids [{'price': '0.012', 'size': '18894.96'}, {'price': '0.011', 'size': '415.28'}, {'price': '0.01', 'size': '620'}]
asks [{'price': '0.013', 'size': '1858.95'}, {'price': '0.014', 'size': '2463.62'}, {'price': '0.017', 'size': '1565.85'}]
tick size 0.001 | min order 5


## Price history

`price_history` takes either an `interval` or a `start_ts`/`end_ts` pair in
unix seconds, and `fidelity` sets the resolution in minutes. With
`as_frame=True` you get a time-indexed DataFrame instead of a list of dicts.

In [6]:
history = market.price_history(interval = "1d", as_frame = True)
history.tail()

,price
t,
2026-08-25 22:43:13+00:00,0.0125
2026-08-25 22:44:14+00:00,0.0125
2026-08-25 22:45:15+00:00,0.0125
2026-08-25 22:46:14+00:00,0.0125
2026-08-25 22:46:14+00:00,0.0125


One quirk worth knowing: the API rejects the longer intervals unless you also
widen `fidelity`. This is the API's rule, not the wrapper's, and it shows up
as a `PolymarketAPIError` carrying the server's complaint.

In [7]:
try:
    market.price_history(interval = "1w")
except pyoly.PolymarketAPIError as error:
    print(error)

https://clob.polymarket.com/prices-history returned HTTP 400: {"error":"invalid filters: minimum 'fidelity' for '1w' range is 5"}


In [8]:
market.price_history(interval = "1w", fidelity = 60, as_frame = True).head()

,price
t,
2026-08-18 23:00:15+00:00,0.0015
2026-08-19 00:00:20+00:00,0.0015
2026-08-19 01:00:18+00:00,0.0015
2026-08-19 02:00:14+00:00,0.0015
2026-08-19 03:00:17+00:00,0.0015


## The clob module directly

The `Market` helpers above cover one outcome at a time. `pyoly.clob` is the
full surface, and it works in token ids, which is how the CLOB names things.

In [9]:
from pyolymarket import clob

yes_token, no_token = market.token_ids
print(yes_token)
print(no_token)

18258395755041392118198940898057846234774760894152262988568951221464458542850
114727316628172873125190766157418606846347212023742363393254518987604640648589


The batch endpoints take a list and answer in one request, keyed by token id.
Note these are POSTs: the documented GET forms answer 400/405.

In [10]:
print(clob.midpoints([yes_token, no_token]))
print(clob.spreads([yes_token, no_token]))
print(clob.prices([(yes_token, "BUY"), (no_token, "BUY")]))

{'114727316628172873125190766157418606846347212023742363393254518987604640648589': '0.9875', '18258395755041392118198940898057846234774760894152262988568951221464458542850': '0.0125'}


{'114727316628172873125190766157418606846347212023742363393254518987604640648589': '0.001', '18258395755041392118198940898057846234774760894152262988568951221464458542850': '0.001'}


{'114727316628172873125190766157418606846347212023742363393254518987604640648589': {'BUY': '0.987'}, '18258395755041392118198940898057846234774760894152262988568951221464458542850': {'BUY': '0.012'}}


In [11]:
clob.last_trade_prices([yes_token, no_token])

[{'price': '0.988',
  'side': 'BUY',
  'token_id': '114727316628172873125190766157418606846347212023742363393254518987604640648589'},
 {'price': '0.012',
  'side': 'SELL',
  'token_id': '18258395755041392118198940898057846234774760894152262988568951221464458542850'}]

`books` and `batch_price_history` follow the same shape. The latter caps out
at 20 tokens per call.

In [12]:
for depth in clob.books([yes_token, no_token]):
    print(depth["asset_id"][:12], "...", len(depth["bids"]), "bids /", len(depth["asks"]), "asks")

182583957550 ... 8 bids / 73 asks
114727316628 ... 73 bids / 8 asks


In [13]:
histories = clob.batch_price_history([yes_token, no_token], interval = "1d")

for token, points in histories.items():
    print(token[:12], "...", len(points), "points, last =", points[-1])

114727316628 ... 1441 points, last = {'t': 1787698093, 'p': 0.9875}
182583957550 ... 1441 points, last = {'t': 1787698093, 'p': 0.0125}


## Market metadata

The CLOB's own view of a market is keyed by condition id, and its `tokens`
list is the authoritative outcome-to-token mapping.

In [14]:
detail = clob.market(market.condition_id)

for token in detail["tokens"]:
    print(token)

print()
print("accepting orders:", detail["accepting_orders"], "| closed:", detail["closed"])

{'token_id': '18258395755041392118198940898057846234774760894152262988568951221464458542850', 'outcome': 'Yes', 'price': 0.0125, 'winner': False}
{'token_id': '114727316628172873125190766157418606846347212023742363393254518987604640648589', 'outcome': 'No', 'price': 0.9875, 'winner': False}

accepting orders: True | closed: False


`market_by_token` goes the other way, from a token id back to the market it
belongs to.

In [15]:
clob.market_by_token(yes_token)

{'condition_id': '0x39ea1503427df7f1439e7dd6567aad5ed72284ed49c58d09e5b917a24389fab7',
 'primary_token_id': '114727316628172873125190766157418606846347212023742363393254518987604640648589',
 'secondary_token_id': '18258395755041392118198940898057846234774760894152262988568951221464458542850'}

In [16]:
print("tick size", clob.tick_size(yes_token))
print("neg risk ", clob.neg_risk(yes_token))
print("server up", clob.ok())
print("server ts", clob.server_time())

tick size 0.001
neg risk  False
server up True


server ts 1787698109


`markets()` pages through the whole book of markets. Feed `next_cursor` back
in to walk forward; the list ends when the cursor comes back as `"LTE="`.

In [17]:
page = clob.markets()
print(len(page["data"]), "markets in this page, next cursor:", page["next_cursor"])

page_two = clob.markets(next_cursor = page["next_cursor"])
print(len(page_two["data"]), "markets in the next page")

1000 markets in this page, next cursor: aWQ6MjQ5Mzk2


1000 markets in the next page


## Authenticated reads

The rest of the module needs L2 credentials in `PYOLY_CLOB_API_KEY`,
`PYOLY_CLOB_SECRET`, `PYOLY_CLOB_PASSPHRASE` and `PYOLY_CLOB_ADDRESS`, so it
is described here rather than run:

- `clob.open_orders()` and `clob.user_trades()` are your own orders and fills.
  There is no public trade feed on the CLOB; for everyone's trades use the
  separate Data API at `data-api.polymarket.com`.
- `clob.balance_allowance()` is your collateral or token balance.
- `clob.order_scoring(order_id)` reports whether an order earns rewards.

`clob.create_api_key()` and `clob.derive_api_key()` mint or re-derive those L2
credentials, and are the only calls that sign with the wallet key in
`PYOLY_CLOB_PRIVATE_KEY`. They need the optional `trading` extra for
`eth-account`. Order placement is out of scope for now; this module reads
only.